In [ ]:
# Setup: clone the repo and switch to the `cmkd` branch (all CMKD code lives there, not on main)
!git clone https://github.com/TotallyNotMinh/Environmental-Sound-Recognition-Under-Unseen-Conditions.git
%cd /kaggle/working/Environmental-Sound-Recognition-Under-Unseen-Conditions
!git fetch origin
!git checkout cmkd
!git pull origin cmkd

In [ ]:
!pip install -q h5py scipy opencv-python tqdm

In [ ]:
# Quick check (~2 min, no dataset needed): CNN training + evaluation on synthetic data.
!python scripts/train_cnn_classifier.py --mock --num-epoch 2 --num-workers 0 --checkpoint-dir /kaggle/working/smoke/cnn/
!python scripts/evaluate.py --mock --checkpoint /kaggle/working/smoke/cnn/best_cnn.pth --num-workers 0

In [ ]:
# Train the CNN baseline (EfficientNet, no KD). CMKD FSD50K CNN recipe: LR 5e-4, batch 24, BCE; 30 epochs here.
# Set ARCH to "b0" or "b4" and run the notebook once per arch (separate Kaggle versions).
# Single process / 1 GPU (this Kaggle image's torch has no NCCL, so torchrun DDP fails). --batch-size 24 is the full batch.
# If CUDA OOM (B4): add --grad-accum-steps 2.
#
# Kaggle stops a session at 12 h, so training is capped at 11 h (timeout) to leave time for evaluation
# and for Kaggle to save /kaggle/working as this version's output.
# To CONTINUE an unfinished run: "Add Input" -> this notebook's previous version output (ONLY the latest one),
# then run again. The cell below finds that checkpoint, copies it here, and training resumes from the last epoch.
import os, shutil

ARCH = "b0"   # "b0" or "b4"
EPOCHS = 30
RUN = f"/kaggle/working/checkpoints/fsd50k_cnn_{ARCH}"
FSD50K_DIR = "/kaggle/input/datasets/yousirui1/fsd50k"   # skipped: ~50k audio files, makes the search slow

def find_in_inputs(name, parent=None):
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if root.startswith(FSD50K_DIR):
            dirs[:] = []
            continue
        dirs[:] = [d for d in dirs if d != "smoke"]   # quick-check models trained on fake data
        if name in files and (parent is None or os.path.basename(root) == parent):
            hits.append(os.path.join(root, name))
    return sorted(hits)

os.makedirs(RUN, exist_ok=True)
prev = find_in_inputs("checkpoint.pth", parent=f"fsd50k_cnn_{ARCH}")
if prev and not os.path.exists(f"{RUN}/checkpoint.pth"):
    src = os.path.dirname(prev[0])
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), RUN)
    print(f"Resuming from previous run: {src}")
elif os.path.exists(f"{RUN}/checkpoint.pth"):
    print("Checkpoint already in /kaggle/working -- resuming.")
else:
    print("No previous checkpoint found -- starting fresh.")

!timeout --signal=INT --kill-after=5m 11h python scripts/train_cnn_classifier.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --arch {ARCH} \
        --batch-size 24 \
        --lr 5e-4 \
        --num-epoch {EPOCHS} \
        --checkpoint-dir {RUN}/ \
        --checkpoint-path {RUN}/checkpoint.pth

In [ ]:
# CNN performance: score the best checkpoint on the FSD50K EVAL split (the split the CMKD paper reports).
# Results are printed and saved to {RUN}/eval_eval_best_cnn.json -- compare this with the AST / KD-AST runs.
import pandas as pd

hist = pd.read_csv(f"{RUN}/metrics.csv")
last = int(hist["epoch"].max())
print(f"Epochs finished: {last}/{EPOCHS}" + ("" if last >= EPOCHS else "  <-- unfinished: add this version's output as input and re-run to continue"))
print(hist.tail(5).to_string(index=False))

!python scripts/evaluate.py \
        --checkpoint {RUN}/best_cnn.pth \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --split eval